# 10 — Colab'da tam değerlendirme (ablasyon + eval)

Ablasyon tablosu jüri için **en ikna edici tek artefakttır** (CLAUDE.md §16).
Bu notebook onu A100 üzerinde gerçek bir LLM ile üretir ve sonuçları zip'ler.

## Neden `LLM_STRICT=1` ile koşuyoruz

Eski kodda LLM hatası sessizce yutuluyordu; ablasyon `hibrit = kural-only`
satırını **hata vermeden** basabiliyordu. Yani tablo, LLM hiç çalışmamışken de
"çalıştı ama katkı vermedi" gibi okunuyordu — jüriye gösterilecek en önemli
artefakt sessizce yanıltıcı olabiliyordu.

`LLM_STRICT=1` bu ihtimali ortadan kaldırır: LLM'e ulaşılamıyorsa, şema
reddediliyorsa ya da çıktı ayrıştırılamıyorsa koşu **exception ile durur**.
Tablo ya gerçektir ya da hiç basılmaz.

## Mimari kural

Tünel/ngrok **yoktur**. vLLM aynı makinede `localhost:8001`'de kalkar; teslim
edilen sistemin Colab'a çalışma zamanı bağımlılığı **olamaz** (şartname §5.9).


In [ ]:
# GPU kontrolü — A100 bekleniyor. Colab'da: Runtime > Change runtime type > A100.
!nvidia-smi


In [ ]:
# vLLM kurulumu (Colab). ~3-5 dk sürer.
# Sürüm SABİTLENMEZ: hangi yapılandırılmış-çıktı parametresinin geçerli olduğu
# sürüme göre değişir ve kod bunu ÖLÇEREK bulur (clients.py yetenek pazarlığı).
# Sabit sürüm, kodun taşınabilirlik iddiasını test etmeden geçirirdi.
!pip install -q vllm
!python -c "import vllm; print('vllm', vllm.__version__)"


In [ ]:
# Depoyu klonla. (Zaten klonluysa bu hücreyi atla ve REPO yolunu elle ayarla.)
import os, pathlib, subprocess

REPO_URL = os.environ.get("ANATOLIA_REPO_URL", "")  # ör. https://github.com/<kullanici>/<repo>.git
REPO = pathlib.Path("/content/anatoliaaI/app")

if not REPO.exists():
    assert REPO_URL, "REPO_URL'i doldurun ya da depoyu elle yükleyin."
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/anatoliaaI"],
                   check=True)

os.chdir(REPO)
print("cwd:", os.getcwd())


In [ ]:
# ---------------------------------------------------------------------------
# Model seçimi — lisans kısıtı MUTLAKTIR (şartname §5.10, CLAUDE.md §7/§20).
# Yalnız Apache-2.0 / MIT ağırlıklar. Qwen3 ailesi Apache-2.0'dır.
# Trendyol-LLM-8B-T1 BLOKELİ: taban model zinciri doğrulanana dek kullanılmaz
# (bkz. docs/model-license-audit.md).
#
# Kalite tavanı (A100 40GB'de deney için):
#   "Qwen/Qwen3-32B"        -> en yüksek kalite, AWQ/kısa bağlam gerekebilir
#   "Qwen/Qwen3-14B-AWQ"    -> 4-bit, A100'e rahat sığar, hızlı
#   "Qwen/Qwen3-8B"         -> güvenli varsayılan
# Teslim/demo (CPU, GPU yok): Qwen3-4B GGUF + Ollama (bkz. 'CPU demo' bölümü).
#
# NOT: depo adlarını çalıştırmadan önce huggingface.co üzerinde doğrulayın;
# nicemlenmiş (AWQ) varyantların adları zamanla değişebilir.
# ---------------------------------------------------------------------------
MODEL = "Qwen/Qwen3-8B"
PORT = 8001                 # 8000 API'nin kendi portu; çakışmasın diye 8001
MAX_LEN = 8192              # 6 few-shot örneği + uzun kampanya metni sığsın


In [ ]:
# vLLM'i AYNI MAKİNEDE, localhost'ta bir alt süreç (subprocess) olarak kaldır.
#
# ÖNEMLİ — mimari kural: burada ngrok/cloudflared/tünel YOKTUR ve olmayacaktır.
# Şartname §5.9 sistemin dış servislere bağımlı olmadan çalışmasını istiyor.
# Colab yalnızca bir KOŞUCUdur (runner); teslim edilen sistem Colab'a bağlanmaz.
# Yerel makine ile Colab arasındaki tek fark bir ORTAM DEĞİŞKENİDİR (VLLM_URL).
import subprocess, sys, time, urllib.request, os

log = open("/content/vllm.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL,
     "--port", str(PORT),
     "--max-model-len", str(MAX_LEN),
     "--gpu-memory-utilization", "0.90"],
    stdout=log, stderr=subprocess.STDOUT)

def wait_ready(port, timeout_s=1200):
    """Model yüklenene kadar bekle. İlk indirme dakikalar sürebilir."""
    started = time.time()
    while time.time() - started < timeout_s:
        if server.poll() is not None:
            raise RuntimeError("vLLM süreci öldü — /content/vllm.log dosyasına bakın")
        try:
            with urllib.request.urlopen(f"http://localhost:{port}/health", timeout=2):
                return time.time() - started
        except Exception:
            time.sleep(5)
    raise TimeoutError("vLLM zamanında ayağa kalkmadı")

print(f"hazır ({wait_ready(PORT):.0f} sn)")
os.environ["VLLM_URL"] = f"http://localhost:{PORT}"
os.environ["VLLM_MODEL"] = MODEL
os.environ["LLM_BACKEND"] = "vllm"


## 1) Kısa akıl sağlığı kontrolü

Ablasyonu 20 dakika koşturup sonunda "bağlanamadım" görmemek için önce tek
çağrılık kontrol: hangi mod seçildi, cevap geliyor mu?


In [ ]:
import os
os.environ["LLM_STRICT"] = "1"       # sessiz düşme YOK

from src.extraction.llm.extractor import default_extractor

ex = default_extractor()             # LLM_BACKEND=vllm
r = ex.call("İhtiyaç finansmanında ilk 6 ay masrafsız, 36 ay vade imkânı.")
print("mod:", r.structured_mode, "| hata:", r.error, "| alan:", len(r.fields))
assert r.error is None, r.error


## 2) Ablasyon: kural-only vs LLM-only vs hibrit

Çıktı hem ekrana hem `eval/reports/` altına yazılır (zip'lenecek).


In [ ]:
import pathlib, subprocess, os, datetime

REPORTS = pathlib.Path("eval/reports")
REPORTS.mkdir(parents=True, exist_ok=True)
damga = datetime.datetime.now().strftime("%Y%m%d-%H%M")

env = dict(os.environ, LLM_BACKEND="vllm", LLM_STRICT="1", PYTHONUNBUFFERED="1")

def kostur(ad, argv):
    p = subprocess.run(argv, env=env, capture_output=True, text=True)
    cikti = p.stdout + ("\n[STDERR]\n" + p.stderr if p.stderr.strip() else "")
    yol = REPORTS / f"{damga}-{ad}.txt"
    yol.write_text(cikti, encoding="utf-8")
    print(f"===== {ad} (dönüş kodu {p.returncode}) =====")
    print(cikti)
    return p.returncode

rc = kostur("ablation", ["python", "-m", "eval.ablation",
                         "--gold", "data/gold/gold.sample.json"])
assert rc == 0, "ablasyon başarısız — STRICT modda hata yutulmaz, yukarıdaki izi okuyun"


## 3) Alan bazlı P/R/F1 (eval)

`eval/run_eval.py` şu an **kural katmanını** ölçer (LLM'i çağırmaz); ablasyondaki
`hibrit` satırıyla karşılaştırmak için taban çizgisi (baseline) verir. Aşırı
iddia etmemek için bu ayrım rapora da böyle geçmelidir.

`--gold` bir DOSYA bekler (klasör değil).


In [ ]:
kostur("run_eval", ["python", "-m", "eval.run_eval",
                    "--gold", "data/gold/gold.sample.json"])


## 4) Koşu künyesi (provenance)

Tablonun hangi model, hangi mod ve hangi kod sürümüyle üretildiği raporun
yanına yazılır. Bu olmadan sayı tekrar-üretilebilir değildir.


In [ ]:
import json, subprocess

kunye = {
    "tarih": damga,
    "model": os.environ.get("VLLM_MODEL"),
    "structured_mode": ex.structured_mode,
    "llm_backend": os.environ.get("LLM_BACKEND"),
    "llm_strict": os.environ.get("LLM_STRICT"),
    "max_model_len": MAX_LEN,
    "git_commit": subprocess.run(["git", "rev-parse", "HEAD"],
                                 capture_output=True, text=True).stdout.strip(),
    "cagri_istatistikleri": ex.summary(),
    "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip(),
}
(REPORTS / f"{damga}-kunye.json").write_text(
    json.dumps(kunye, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(kunye, ensure_ascii=False, indent=2))


## 5) Raporları zip'le ve indir

Sonuçlar Colab'da kalmaz; repoya/teslime iner. Colab'a bağımlılık burada da yok:
zip yerel makinede açılır, `eval/reports/` altına konur.


In [ ]:
import shutil
zip_yolu = shutil.make_archive(f"/content/eval-reports-{damga}", "zip", REPORTS)
print("zip:", zip_yolu)

try:
    from google.colab import files
    files.download(zip_yolu)
except ImportError:
    print("Colab dışındasınız — zip'i yukarıdaki yoldan alın.")


In [ ]:
# Sunucuyu kapat (GPU belleğini bırak).
server.terminate()
server.wait(timeout=60)
print("vLLM kapandı")
